# Lesson 04 - Padrão de Design de Utilização de Ferramentas

Nesta lição irá aprender o padrão de design **Utilização de Ferramentas** para agentes de IA utilizando o Microsoft Agent Framework (Python). Abrangemos:

- Definir ferramentas de função com o decorador `@tool` e parâmetros tipados
- Fornecer esquemas de ferramentas para que o modelo saiba o que cada ferramenta faz
- Controlar a execução da ferramenta com `approval_mode`
- Retornar **saída estruturada** através de modelos Pydantic e `response_format`

O cenário é um **agente de reservas de viagens** que pode pesquisar destinos, verificar disponibilidade e obter informações de voos.


## Configuração


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Definindo Ferramentas com o Decorador @tool

O decorador `@tool` transforma uma função Python simples numa ferramenta que um agente pode chamar.
Pontos-chave:

- A **docstring** torna-se a descrição da ferramenta que o modelo vê.
- As **anotações de tipo** (incluindo `Annotated` com descrições) definem o esquema da ferramenta.
- `approval_mode` controla se o utilizador deve aprovar cada chamada antes de ser executada.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Criar um Agente com Múltiplas Ferramentas

Passe as três ferramentas para o cliente para que o modelo possa invocar aquelas de que necessita para responder à pergunta do utilizador.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Saída Estruturada com Ferramentas

Ao definir `response_format` para um modelo Pydantic, o agente é obrigado a retornar um objeto JSON bem tipado em vez de texto livre. Isto é útil quando o código subsequente precisa consumir o resultado programaticamente.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Padrões de Aprovação de Ferramentas

O parâmetro `approval_mode` no `@tool` controla se as chamadas às ferramentas requerem aprovação humana antes de serem executadas:

| Modo | Comportamento |
|---|---|
| `"never_require"` | A ferramenta é executada automaticamente — não é necessária confirmação do utilizador. |
| `"always_require"` | Cada chamada deve ser aprovada pelo utilizador antes de ser executada. |

Use `"always_require"` para ferramentas que tenham efeitos secundários (por exemplo, reservar um voo, cobrar um cartão de crédito) para que um humano permaneça no processo.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Sumário

Nesta lição aprendeu a:

1. **Definir ferramentas** usando o decorador `@tool` com parâmetros tipados e docstrings que servem como esquema da ferramenta.
2. **Compor múltiplas ferramentas** para que o agente possa chamá-las em sequência para responder a consultas complexas.
3. **Retornar saída estruturada** passando um modelo Pydantic como `response_format`.
4. **Controlar a aprovação da ferramenta** com `approval_mode` para manter um humano envolvido em operações sensíveis.

Estes padrões formam a base para construir agentes fiáveis e prontos para produção que podem interagir com sistemas externos de forma segura.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:  
Este documento foi traduzido utilizando o serviço de tradução por IA [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original, no seu idioma nativo, deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se a tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas decorrentes do uso desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
